# 2025 기계론적 해석성: SAE·Attribution Graph 실습 - Sparse Feature Decomposition Toy SAE

- Tutorial ID: `mod-interpretability-2025-lab`
- Tutorial: 2025 기계론적 해석성: SAE·Attribution Graph 실습
- Section ID: `interp-1`
- Section: Sparse Feature Decomposition Toy SAE


In [ ]:
# ============================================================
# 코드 읽는 법 — Sparse Feature Decomposition Toy SAE
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(8)
# Toy activation = sparse latent features mixed into residual stream
n_features, d = 6, 4
D = np.random.randn(n_features, d)
D = D / np.linalg.norm(D, axis=1, keepdims=True)
z_true = np.array([1.5,0,0.8,0,0,1.2])
x = z_true @ D + 0.05*np.random.randn(d)
# simple matching pursuit sparse coding
res=x.copy(); z=np.zeros(n_features)
for _ in range(3):
    corr=D@res; i=int(np.argmax(np.abs(corr)))
    z[i]+=corr[i]; res-=corr[i]*D[i]
print('true sparse features:', z_true)
print('recovered approx   :', np.round(z,2))
print('reconstruction err :', round(np.linalg.norm(res),3))
active=np.where(np.abs(z)>0.1)[0].tolist()
print('attribution graph nodes = active features', active)